# Zwesta Trade Performance Analysis

This notebook can analyze your stored trade history from either:

1. the local SQLite database, or
2. the authenticated backend API after login.

## Login

If you use `SOURCE_MODE = 'db'`, you do **not** need to log in.

If you use `SOURCE_MODE = 'api'`, the notebook will log in through `POST /api/user/login`, read the returned `session_token`, and send it in the `X-Session-Token` header while fetching trade history from `/api/bot/status?include_history=true`.

In [ ]:
from pathlib import Path
import json
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

## Configure Data Source

Adjust the filters below if you want to limit analysis to a single user, bot, or date range.

In [ ]:
PROJECT_ROOT = Path.cwd()
DB_CANDIDATES = [
    PROJECT_ROOT / 'backend.db',
    PROJECT_ROOT / 'zwesta_trader.db',
    PROJECT_ROOT / 'zwesta_trading.db',
    PROJECT_ROOT / 'broker_manager.db',
]

DB_PATH = next((path for path in DB_CANDIDATES if path.exists()), None)

SOURCE_MODE = 'db'          # 'db' or 'api'
USER_ID_FILTER = None       # used in db mode after load
BOT_ID_FILTER = None        # used in db mode and api mode
START_DATE = None           # example: '2026-01-01'
END_DATE = None             # example: '2026-12-31'
STARTING_CAPITAL = 10000.0  # used for return estimates and pyfolio

BASE_URL = 'http://127.0.0.1:5000'
EMAIL = 'your-email@example.com'
PASSWORD = 'your-password'
API_MODE_FILTER = ''        # '', 'LIVE', or 'DEMO'

print(f'SOURCE_MODE={SOURCE_MODE}')
if DB_PATH is not None:
    print(f'Local database available: {DB_PATH}')
else:
    print('No local database detected in the current folder')

In [ ]:
def table_exists(conn: sqlite3.Connection, table_name: str) -> bool:
    row = conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (table_name,)).fetchone()
    return row is not None

def login_backend(base_url: str, email: str, password: str) -> dict:
    import requests
    response = requests.post(
        f'{base_url}/api/user/login',
        json={'email': email, 'password': password},
        timeout=20,
    )
    response.raise_for_status()
    payload = response.json()
    if not payload.get('success'):
        raise RuntimeError(payload.get('error') or 'Login failed')
    if payload.get('requires_2fa'):
        raise RuntimeError('2FA is required for this account. Complete 2FA in the app or add a notebook flow for /api/user/verify-2fa.')
    if not payload.get('session_token'):
        raise RuntimeError('Login succeeded but no session_token was returned.')
    return payload

def load_trades_from_db(db_path: Path) -> pd.DataFrame:
    if db_path is None:
        raise FileNotFoundError('No local database file was found for db mode.')
    conn = sqlite3.connect(db_path)
    try:
        if not table_exists(conn, 'trades'):
            raise ValueError('The trades table was not found in the selected database.')

        query = '''
        SELECT
            trade_id,
            bot_id,
            user_id,
            symbol,
            order_type,
            volume,
            price,
            profit,
            commission,
            swap,
            ticket,
            time_open,
            time_close,
            status,
            created_at,
            updated_at,
            trade_data,
            timestamp
        FROM trades
        ORDER BY COALESCE(time_close, time_open, created_at, updated_at) ASC
        '''
        return pd.read_sql_query(query, conn)
    finally:
        conn.close()

def load_trades_from_api(base_url: str, email: str, password: str, mode_filter: str = '', bot_id_filter: str = None) -> pd.DataFrame:
    import requests

    login_payload = login_backend(base_url, email, password)
    headers = {'X-Session-Token': login_payload['session_token']}
    params = {'include_history': 'true'}
    if mode_filter:
        params['mode'] = str(mode_filter).upper()
    if bot_id_filter:
        params['bot_id'] = bot_id_filter

    response = requests.get(
        f'{base_url}/api/bot/status',
        headers=headers,
        params=params,
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if not payload.get('success'):
        raise RuntimeError(payload.get('error') or 'Failed to fetch bot status')

    records = []
    for bot in payload.get('bots', []):
        bot_id = bot.get('botId')
        trade_history = bot.get('tradeHistory') or []
        for trade in trade_history:
            if not isinstance(trade, dict):
                continue
            records.append({
                'trade_id': str(trade.get('ticket') or trade.get('tradeId') or ''),
                'bot_id': bot_id,
                'user_id': login_payload.get('user_id'),
                'symbol': trade.get('symbol'),
                'order_type': trade.get('type') or trade.get('order_type'),
                'volume': trade.get('volume', 0.0),
                'price': trade.get('entryPrice', trade.get('openPrice', trade.get('price', 0.0))),
                'profit': trade.get('profit', 0.0),
                'commission': trade.get('commission', 0.0),
                'swap': trade.get('swap', 0.0),
                'ticket': trade.get('ticket'),
                'time_open': trade.get('entryTime', trade.get('time_open', trade.get('time'))),
                'time_close': trade.get('exitTime', trade.get('time_close')),
                'status': trade.get('status', 'closed' if trade.get('exitTime') or trade.get('time_close') else 'open'),
                'created_at': trade.get('time', trade.get('entryTime')),
                'updated_at': trade.get('exitTime', trade.get('time')),
                'trade_data': json.dumps(trade),
                'timestamp': trade.get('timestamp'),
            })

    return pd.DataFrame(records)

if SOURCE_MODE == 'api':
    raw_trades = load_trades_from_api(BASE_URL, EMAIL, PASSWORD, API_MODE_FILTER, BOT_ID_FILTER)
else:
    raw_trades = load_trades_from_db(DB_PATH)

raw_trades.head()

In [ ]:
def parse_trade_data(value):
    if not value:
        return {}
    if isinstance(value, dict):
        return value
    try:
        decoded = json.loads(value)
        return decoded if isinstance(decoded, dict) else {}
    except Exception:
        return {}

def normalize_trades(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    normalized = df.copy()
    extras = normalized['trade_data'].apply(parse_trade_data)

    normalized['entry_price'] = [extra.get('entryPrice', extra.get('openPrice', price)) for extra, price in zip(extras, normalized['price'])]
    normalized['broker'] = [extra.get('broker') for extra in extras]
    normalized['close_reason'] = [extra.get('closeReason') for extra in extras]

    normalized['opened_at'] = pd.to_datetime(normalized['time_open'], errors='coerce')
    normalized['closed_at'] = pd.to_datetime(normalized['time_close'], errors='coerce')
    normalized['created_at_dt'] = pd.to_datetime(normalized['created_at'], errors='coerce')
    normalized['event_time'] = normalized['closed_at'].combine_first(normalized['opened_at']).combine_first(normalized['created_at_dt'])

    normalized['profit'] = pd.to_numeric(normalized['profit'], errors='coerce').fillna(0.0)
    normalized['commission'] = pd.to_numeric(normalized['commission'], errors='coerce').fillna(0.0)
    normalized['swap'] = pd.to_numeric(normalized['swap'], errors='coerce').fillna(0.0)
    normalized['volume'] = pd.to_numeric(normalized['volume'], errors='coerce').fillna(0.0)
    normalized['price'] = pd.to_numeric(normalized['price'], errors='coerce').fillna(0.0)
    normalized['entry_price'] = pd.to_numeric(normalized['entry_price'], errors='coerce').fillna(normalized['price'])
    normalized['net_profit'] = normalized['profit'] + normalized['commission'] + normalized['swap']

    if USER_ID_FILTER:
        normalized = normalized[normalized['user_id'] == USER_ID_FILTER]
    if BOT_ID_FILTER:
        normalized = normalized[normalized['bot_id'] == BOT_ID_FILTER]
    if START_DATE:
        normalized = normalized[normalized['event_time'] >= pd.Timestamp(START_DATE)]
    if END_DATE:
        normalized = normalized[normalized['event_time'] <= pd.Timestamp(END_DATE)]

    normalized = normalized.sort_values('event_time').reset_index(drop=True)
    return normalized

trades = normalize_trades(raw_trades)
closed_trades = trades[(trades['status'].fillna('').str.lower() == 'closed') | trades['closed_at'].notna()].copy()
print(f'Loaded {len(trades):,} trades total from {SOURCE_MODE} mode')
print(f'Closed trades used for analytics: {len(closed_trades):,}')
closed_trades.head()

In [ ]:
if closed_trades.empty:
    raise ValueError('No closed trades matched the current filters. Adjust USER_ID_FILTER, BOT_ID_FILTER, or date range.')

equity_curve = closed_trades[['event_time', 'net_profit']].copy()
equity_curve['cumulative_profit'] = equity_curve['net_profit'].cumsum()
equity_curve['equity'] = STARTING_CAPITAL + equity_curve['cumulative_profit']
equity_curve['peak_equity'] = equity_curve['equity'].cummax()
equity_curve['drawdown'] = equity_curve['equity'] - equity_curve['peak_equity']
equity_curve['drawdown_pct'] = np.where(equity_curve['peak_equity'] > 0, equity_curve['drawdown'] / equity_curve['peak_equity'], 0.0)

wins = closed_trades[closed_trades['net_profit'] > 0]['net_profit']
losses = closed_trades[closed_trades['net_profit'] < 0]['net_profit']
daily_pnl = closed_trades.set_index('event_time')['net_profit'].resample('D').sum().fillna(0.0)
estimated_daily_returns = (daily_pnl / STARTING_CAPITAL).replace([np.inf, -np.inf], np.nan).fillna(0.0)

summary = pd.Series({
    'trade_count': int(len(closed_trades)),
    'win_rate_pct': round((closed_trades['net_profit'] > 0).mean() * 100, 2),
    'net_profit': round(closed_trades['net_profit'].sum(), 2),
    'gross_profit': round(wins.sum(), 2),
    'gross_loss': round(losses.sum(), 2),
    'profit_factor': round(wins.sum() / abs(losses.sum()), 3) if len(losses) else np.nan,
    'avg_trade': round(closed_trades['net_profit'].mean(), 2),
    'avg_win': round(wins.mean(), 2) if len(wins) else 0.0,
    'avg_loss': round(losses.mean(), 2) if len(losses) else 0.0,
    'best_trade': round(closed_trades['net_profit'].max(), 2),
    'worst_trade': round(closed_trades['net_profit'].min(), 2),
    'max_drawdown': round(equity_curve['drawdown'].min(), 2),
    'max_drawdown_pct': round(equity_curve['drawdown_pct'].min() * 100, 2),
    'ending_equity': round(equity_curve['equity'].iloc[-1], 2),
    'return_on_starting_capital_pct': round((equity_curve['equity'].iloc[-1] / STARTING_CAPITAL - 1) * 100, 2),
    'first_trade': str(closed_trades['event_time'].min()),
    'last_trade': str(closed_trades['event_time'].max()),
})
summary.to_frame('value')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 14), constrained_layout=True)

axes[0].plot(equity_curve['event_time'], equity_curve['equity'], color='#0b84a5', linewidth=2)
axes[0].set_title('Estimated Equity Curve')
axes[0].set_ylabel('Equity')

axes[1].fill_between(equity_curve['event_time'], equity_curve['drawdown_pct'] * 100, 0, color='#f95d6a', alpha=0.35)
axes[1].set_title('Drawdown (%)')
axes[1].set_ylabel('Drawdown %')

daily_pnl.plot(ax=axes[2], kind='bar', color=np.where(daily_pnl >= 0, '#2ca02c', '#d62728'), width=0.9)
axes[2].set_title('Daily Net PnL')
axes[2].set_ylabel('PnL')
axes[2].tick_params(axis='x', rotation=45)

plt.show()

In [ ]:
closed_trades[['event_time', 'bot_id', 'user_id', 'symbol', 'order_type', 'volume', 'entry_price', 'net_profit', 'broker', 'close_reason']]
    .sort_values('event_time', ascending=False)
    .head(25)

## Optional Pyfolio Tear Sheet

This uses estimated daily returns based on `STARTING_CAPITAL`. If you want institution-grade analytics, the better input is a real daily equity/NAV time series from the broker.

In [ ]:
try:
    import pyfolio as pf
    print('pyfolio is installed')
except ImportError:
    pf = None
    print('pyfolio is not installed. Run: pip install pyfolio')

In [ ]:
if pf is not None:
    returns = estimated_daily_returns.copy()
    returns.index = pd.to_datetime(returns.index).tz_localize(None)
    pf.create_simple_tear_sheet(returns)
else:
    print('Skip this cell until pyfolio is installed.')

## API Mode Usage

Set `SOURCE_MODE = 'api'` in Cell 4, then fill in:

- `BASE_URL`
- `EMAIL`
- `PASSWORD`
- optional `API_MODE_FILTER` with `LIVE` or `DEMO`
- optional `BOT_ID_FILTER` for one bot

When you run Cell 5, the notebook will:

1. log in with `POST /api/user/login`,
2. collect the `session_token`,
3. call `GET /api/bot/status?include_history=true`, and
4. flatten each bot's `tradeHistory` into the same analytics pipeline used for DB mode.

In [ ]:
# Quick login check for api mode
if SOURCE_MODE != 'api':
    print("Set SOURCE_MODE = 'api' in Cell 4 if you want to test backend login.")
else:
    login_payload = login_backend(BASE_URL, EMAIL, PASSWORD)
    {key: login_payload.get(key) for key in ['success', 'user_id', 'email', 'name', 'session_token']}

In [ ]:
# Example protected request after login
if SOURCE_MODE != 'api':
    print("This example only applies in api mode.")
else:
    headers = {'X-Session-Token': login_payload['session_token']}
    import requests
    response = requests.get(
        f'{BASE_URL}/api/bot/status',
        headers=headers,
        params={'include_history': 'true', 'mode': API_MODE_FILTER} if API_MODE_FILTER else {'include_history': 'true'},
        timeout=30,
    )
    response.raise_for_status()
    bot_status_payload = response.json()
    {'success': bot_status_payload.get('success'), 'activeBots': bot_status_payload.get('activeBots'), 'botCount': len(bot_status_payload.get('bots', []))}